# Dropout-risk peer graph pipeline (single-file, corrected — Colab-ready)

Reorganized into one linear run for Colab. Fixes applied (noted inline with `# FIX:`):
- `torch_geometric` is installed **before** anything imports it.
- Removed the duplicate node/edge-tensor construction cell.
- Removed the pointless save -> download -> re-upload -> reload round trip;
  `data` stays in memory and is reused directly by the graph checks/plots.
- Fixed the copy-paste bug where the edges CSV was never downloaded.
- **Fixed a save/download filename mismatch in the export cell**: files were
  being *saved* under `Synthetic_school_data_...` names but *downloaded*
  under `Synthetic_school_data_...` names, which would throw a file-not-found
  error in Colab. Save and download names now match everywhere (`Synthetic_...`).
- Uses `google.colab.files.download` directly (this notebook is meant to run
  in Colab, so no local-vs-Colab branching is needed) — just run top to
  bottom and each artifact will prompt a download as it's produced.
- Upload your data file when prompted in Cell 1 (or place it in the Colab
  file browser first) — the loader looks for `Synthetic school data.xlsx`.

In [ ]:
# ============================================================
# CELL 0 — SETUP: install + imports (torch_geometric BEFORE use)
# ============================================================
!pip install -q torch_geometric

# CONFIG: set this per run so output filenames/plot titles never collide
# or get mislabeled across different subset runs.
SUBSET_TAG = "full"  # e.g. "1000", "3000", "full" -- change this each run

import pandas as pd
import numpy as np
import torch
import networkx as nx
import matplotlib.pyplot as plt
import pickle

from sklearn.preprocessing import StandardScaler
from sklearn.metrics.pairwise import cosine_similarity
from sklearn.model_selection import train_test_split
from itertools import combinations
from scipy.sparse import coo_matrix, save_npz
from torch_geometric.data import Data
from google.colab import files

In [ ]:
# ============================================================
# CELL 1 — UPLOAD + LOAD + FIX grade_number contamination from CTGAN sampling
# ============================================================
# Run this cell, then use the file picker that appears to upload
# 'Synthetic school data.xlsx'. If it's already in the Colab file browser
# (left sidebar), you can skip the upload() call and just read it directly.
uploaded = files.upload()  # select "Synthetic school data.xlsx"

df = pd.read_excel('Synthetic school data.xlsx')
df_raw = df.copy()  # keep an unencoded copy for any future crosstabs/checks

df['grade_number'] = df['grade_number'].round().astype(int)

In [ ]:
# ============================================================
# CELL 2 — ENCODE: binary + ordinal (order preserved)
# ============================================================
df['Gender_enc'] = (df['Gender'] == 'Female').astype(int)

edu_order = {'JHS': 0, 'SHS': 1, 'Tertiary': 2}
df['Parental_edu_enc'] = df['Parental educational level'].map(edu_order)

child_labor_order = {'No': 0, 'Sometimes': 1, 'Yes': 2}
df['Child_labor_enc'] = df['Child labor involvement'].map(child_labor_order)

rel_order = {'Poor': 0, 'Average': 1, 'Good': 2}
df['Teacher_rel_enc'] = df['Teacher relationship quality'].map(rel_order)
df['Peer_rel_enc'] = df['Peer relationship quality'].map(rel_order)
# Household income level (int, 1/2/3) already ordinal — kept as-is

In [ ]:
# ============================================================
# CELL 3 — ENCODE: nominal one-hot
# ============================================================
onehot_cols = ['School', 'Family dropout history', 'Mode of transport',
               'Extra-curricular activities']
df = pd.get_dummies(df, columns=onehot_cols, prefix=onehot_cols, dtype=int)

df['subgroup_cat'] = df['subgroup'].fillna(-1).astype(int).astype(str)
df['stream_letter_cat'] = df['stream_letter'].fillna('none')
df = pd.get_dummies(df, columns=['subgroup_cat', 'stream_letter_cat'],
                     prefix=['subgroup', 'stream'], dtype=int)

In [ ]:
# ============================================================
# CELL 4 — COLLAPSE attendance: level + trend
# FIX: split into early (weeks 1-7, feature) vs late (weeks 8-14,
# label input only). Previously attendance_mean/attendance_trend used
# all 14 weeks and fed BOTH x and the risk_score formula -- meaning
# the label could be reconstructed from a feature the model could see.
# The early period gives the model real, non-leaking signal to learn
# from; the late period is reserved for defining the label, matching
# a genuine early-warning framing (predict later trajectory from
# earlier signal) rather than classifying current state from current
# data.
# ============================================================
week_cols_early = [f'Week{i}_attendance' for i in range(1, 8)]   # weeks 1-7
week_cols_late  = [f'Week{i}_attendance' for i in range(8, 15)]  # weeks 8-14
early_matrix = df[week_cols_early].to_numpy(dtype=float)
late_matrix  = df[week_cols_late].to_numpy(dtype=float)

# early attendance -> goes into x (features)
df['attendance_mean_early'] = early_matrix.mean(axis=1)
early_weeks_centered = np.arange(1, 8) - np.arange(1, 8).mean()
df['attendance_trend_early'] = (early_matrix * early_weeks_centered).sum(axis=1) / (early_weeks_centered ** 2).sum()

# late attendance -> goes into the label formula only, NOT into x
df['attendance_mean_late'] = late_matrix.mean(axis=1)
late_weeks_centered = np.arange(8, 15) - np.arange(8, 15).mean()
df['attendance_trend_late'] = (late_matrix * late_weeks_centered).sum(axis=1) / (late_weeks_centered ** 2).sum()

df = df.drop(columns=week_cols_early + week_cols_late)


In [ ]:
# ============================================================
# CELL 5 — SCALE continuous columns
# ============================================================
cont_cols = ['Semester 1 average', 'Semester 2 average', 'Semester difference',
             'Travel time to school (minutes)']
scaler = StandardScaler()
df[cont_cols] = scaler.fit_transform(df[cont_cols])
# attendance_mean/trend already handled above

In [ ]:
# ============================================================
# CELL 6 — DROP non-feature / redundant columns
# ============================================================
df = df.drop(columns=[
    'Student ID', 'Class level', 'Travel time to school',
    'Household income level (standardized)', 'level_prefix',
    'Gender', 'Parental educational level', 'Child labor involvement',
    'Teacher relationship quality', 'Peer relationship quality',
    'subgroup', 'stream_letter'
])

print("Node feature matrix shape:", df.shape)

In [ ]:
# ============================================================
# CELL 7 — class_group needs School included
# ============================================================
school_key = (
    df['School_Ayeduase RC'].astype(str) + '_' +
    df['School_Shining Star Preparatory'].astype(str) + '_' +
    df['School_Weweso MA'].astype(str)
)

df['class_group'] = (
    school_key + '_' +
    df['grade_number'].astype(str) + '_' +
    df['stream_A'].astype(str) + df['stream_B'].astype(str) + '_' +
    df['subgroup_-1'].astype(str) + df['subgroup_1'].astype(str) +
    df['subgroup_2'].astype(str) + df['subgroup_3'].astype(str)
)

print("Number of unique classroom groups:", df['class_group'].nunique())

In [ ]:
# ============================================================
# CELL 8 — similarity_cols: which columns feed edge similarity
#          (academic outcomes held out so edges aren't circular
#          with the thing being predicted)
# FIX: this list still referenced the pre-split column names
# ('attendance_mean', 'attendance_trend'), which no longer exist after
# CELL 4 renamed them to _early/_late. Because the names didn't match,
# the exclusion silently did nothing -- both attendance_mean_late and
# attendance_trend_late (the columns the label is built from, see
# CELL 12) were still being fed into the peer-similarity calculation,
# meaning the graph structure itself leaked label-defining information.
# This affects only GAT/Gated-GAT (the graph models); it does not
# explain underperformance in Logistic Regression / Random Forest,
# which never see the graph at all.
# Now excludes all four attendance columns (early + late), consistent
# with the original intent of holding out attendance/academic signal
# from edge-building entirely -- not just the late/label-defining half.
# ============================================================
academic_cols = ['attendance_mean_early', 'attendance_trend_early',
                  'attendance_mean_late', 'attendance_trend_late',
                  'Semester 1 average', 'Semester 2 average', 'Semester difference']
non_feature_cols = ['class_group', 'Section']
similarity_cols = [c for c in df.columns if c not in academic_cols + non_feature_cols]


In [ ]:
# ============================================================
# CELL 9 — EDGE CONSTRUCTION (per-classroom cosine similarity + section boost)
# FIX (merged from patched version): minimum-degree fallback -- a student
# can miss the top-1% cutoff purely because their class_group is small
# (fewer pairs = fewer chances to clear a strict percentile), not because
# they're actually dissimilar to everyone. For any student in a group with
# zero edges after the cutoff pass, connect them to their single
# most-similar classmate. This does not change the 99th-percentile rule
# for anyone else -- it only guarantees no student is isolated purely as
# an artifact of small class size.
# FIX: the fallback pass now applies the same section_boost as the main
# cutoff pass -- the patched version's fallback used raw cosine similarity
# with no boost, so same-section pairs were treated inconsistently
# depending on which pass connected them.
# ============================================================
target_percentile = 99  # locked-in value
section_boost = 0.03

edges = []

for group_key, group_df in df.groupby('class_group'):
    if len(group_df) < 2:
        continue

    idx = group_df.index.to_numpy()
    feats = group_df[similarity_cols].to_numpy(dtype=float)
    sections = group_df['Section'].to_numpy()
    sim_matrix = cosine_similarity(feats)

    iu = np.triu_indices(len(feats), k=1)
    pair_sims = sim_matrix[iu]
    if len(pair_sims) == 0:
        continue
    cutoff = np.percentile(pair_sims, target_percentile)

    # boosted similarity matrix -- section_boost applied once, used by
    # both the cutoff pass and the fallback pass so edges are scored
    # consistently regardless of which pass created them
    boosted_sim = sim_matrix.copy()
    same_section = sections[:, None] == sections[None, :]
    boosted_sim[same_section] = np.minimum(boosted_sim[same_section] + section_boost, 1.0)

    group_edges = []  # edges found for THIS group only, before the fallback step
    for i, j in combinations(range(len(idx)), 2):
        sim = boosted_sim[i, j]
        if sim >= cutoff:
            group_edges.append((idx[i], idx[j], sim))

    connected_in_group = set()
    for s, t, _ in group_edges:
        connected_in_group.add(s)
        connected_in_group.add(t)

    for local_i in range(len(idx)):
        if idx[local_i] in connected_in_group:
            continue
        sims_to_others = boosted_sim[local_i].copy()
        sims_to_others[local_i] = -np.inf  # exclude self
        best_j = int(np.argmax(sims_to_others))
        best_sim = boosted_sim[local_i, best_j]
        pair = tuple(sorted((idx[local_i], idx[best_j])))
        group_edges.append((pair[0], pair[1], best_sim))
        connected_in_group.add(idx[local_i])
        connected_in_group.add(idx[best_j])

    edges.extend(group_edges)

edge_df = pd.DataFrame(edges, columns=['source', 'target', 'weight']).drop_duplicates(subset=['source', 'target'])

print(f"Total edges: {len(edge_df)}")
print(f"Students with at least one edge: {edge_df[['source','target']].stack().nunique()} / {len(df)}")
print(f"Avg degree: {2 * len(edge_df) / len(df):.2f}")
print(f"Isolated students: {len(df) - edge_df[['source','target']].stack().nunique()}")


In [ ]:
# ============================================================
# CELL 10 — sanity check: no cross-school edges should exist
# ============================================================
school_cols = ['School_Ayeduase RC', 'School_Shining Star Preparatory', 'School_Weweso MA']
df_school = df[school_cols].idxmax(axis=1)

cross_school_edges = 0
for _, row in edge_df.iterrows():
    if df_school.loc[row['source']] != df_school.loc[row['target']]:
        cross_school_edges += 1

print("Cross-school edges (should be 0):", cross_school_edges)

In [ ]:
# ============================================================
# CELL 11 — add self-loops
# ============================================================
self_loops = pd.DataFrame({
    'source': df.index,
    'target': df.index,
    'weight': 1.0
})
edge_df_with_loops = pd.concat([edge_df, self_loops], ignore_index=True)
print("Edges with self-loops:", len(edge_df_with_loops))

In [ ]:
# ============================================================
# CELL 12 — dropout risk label
# FIX: risk label now built from LATE-period attendance signal only
# (attendance_mean_late / attendance_trend_late), consistent with the
# early/late split introduced in CELL 4.
# ============================================================
def zscore(s):
    return (s - s.mean()) / s.std()

attendance_signal = (zscore(df['attendance_mean_late']) + zscore(df['attendance_trend_late'])) / 2
academic_signal = (zscore(df['Semester 2 average']) + zscore(df['Semester difference'])) / 2

df['risk_score'] = -(0.65 * attendance_signal + 0.35 * academic_signal)

risk_pct = df['risk_score'].rank(pct=True)
df['dropout_risk'] = np.select(
    [risk_pct >= 0.72, risk_pct <= 0.27],
    ['High', 'Low'],
    default='Medium'
)

print(df['dropout_risk'].value_counts())
print(df['dropout_risk'].value_counts(normalize=True).round(3))


In [ ]:
# ============================================================
# CELL 13 — build the PyG Data object
# FIX: the label is a deterministic formula over these four columns
# (see CELL 12) -- leaving them in x lets any model reconstruct the
# label directly, bypassing the graph/task entirely. Excluded here.
# Only the LATE attendance columns are excluded (they feed the label);
# EARLY attendance columns stay in as legitimate, non-leaking features.
# ============================================================
non_feature_cols = ['class_group', 'Section', 'risk_score', 'dropout_risk',
                     'attendance_mean_late', 'attendance_trend_late',
                     'Semester 2 average', 'Semester difference']
# NOTE: attendance_mean_early / attendance_trend_early are intentionally
# left OUT of this exclusion list -- they stay in x as real features.
feature_cols_final = [c for c in df.columns if c not in non_feature_cols]

x = torch.tensor(df[feature_cols_final].to_numpy(dtype=float), dtype=torch.float)

label_map = {'Low': 0, 'Medium': 1, 'High': 2}
y = torch.tensor(df['dropout_risk'].map(label_map).to_numpy(), dtype=torch.long)

print("x shape:", x.shape)
print("y shape:", y.shape)

# --- edge_index, bidirectional ---
reverse_edges = edge_df_with_loops[edge_df_with_loops['source'] != edge_df_with_loops['target']].copy()
reverse_edges = reverse_edges.rename(columns={'source': 'target', 'target': 'source'})[['source', 'target', 'weight']]
edge_df_bidirectional = pd.concat([edge_df_with_loops, reverse_edges], ignore_index=True)

edge_index = torch.tensor(edge_df_bidirectional[['source', 'target']].to_numpy().T, dtype=torch.long)
edge_weight = torch.tensor(edge_df_bidirectional['weight'].to_numpy(), dtype=torch.float)

print("edge_index shape:", edge_index.shape)

# --- stratified 70/15/15 split ---
indices = np.arange(len(y))
train_idx, temp_idx = train_test_split(indices, test_size=0.30, stratify=y.numpy(), random_state=42)
val_idx, test_idx = train_test_split(temp_idx, test_size=0.50, stratify=y.numpy()[temp_idx], random_state=42)

train_mask = torch.zeros(len(y), dtype=torch.bool)
val_mask = torch.zeros(len(y), dtype=torch.bool)
test_mask = torch.zeros(len(y), dtype=torch.bool)
train_mask[train_idx] = True
val_mask[val_idx] = True
test_mask[test_idx] = True

data = Data(x=x, edge_index=edge_index, edge_attr=edge_weight, y=y,
            train_mask=train_mask, val_mask=val_mask, test_mask=test_mask)

print(data)
print("Train:", train_mask.sum().item(), "Val:", val_mask.sum().item(), "Test:", test_mask.sum().item())
for name, mask in [('Train', train_mask), ('Val', val_mask), ('Test', test_mask)]:
    print(name, torch.bincount(y[mask]))

In [ ]:
# ============================================================
# CELL 14 — save + download artifacts
# FIX: save and download filenames now match exactly (previously
# files were saved under one prefix but downloaded under another;
# both are now consistently 'Synthetic_school_data_...' throughout.

# ============================================================
pt_name = f'Synthetic_school_data_{SUBSET_TAG}.pt'
features_csv_name = f'Synthetic_school_data_features_df_{SUBSET_TAG}.csv'
edges_csv_name = f'Synthetic_school_data_edges_df_{SUBSET_TAG}_v2.csv'

torch.save(data, pt_name)
df.to_csv(features_csv_name, index=False)
edge_df_bidirectional.to_csv(edges_csv_name, index=False)

files.download(pt_name)
files.download(features_csv_name)
files.download(edges_csv_name)

In [ ]:
# ============================================================
# CELL 15 — graph checks + plots (reuses `data` already in memory)
# ============================================================
edge_index_np = data.edge_index.numpy()
G_full = nx.Graph()
G_full.add_nodes_from(range(data.x.shape[0]))
edges_no_loops = [(int(s), int(t)) for s, t in zip(edge_index_np[0], edge_index_np[1]) if s != t]
G_full.add_edges_from(edges_no_loops)

components = sorted(nx.connected_components(G_full), key=len, reverse=True)
component_sizes = [len(c) for c in components]
print("Number of components:", len(components))
print("Largest 10 component sizes:", component_sizes[:10])

target_component = next((c for c in components if 15 <= len(c) <= 40), components[0])
print(f"\nSelected component size: {len(target_component)}")

# ============ PLOT 1: one classroom cluster in detail ============
subG = G_full.subgraph(target_component)
label_map_reverse = {0: 'Low', 1: 'Medium', 2: 'High'}
node_colors = [{'Low': 'green', 'Medium': 'orange', 'High': 'red'}[label_map_reverse[data.y[n].item()]]
               for n in subG.nodes()]

plt.figure(figsize=(8, 8))
pos = nx.spring_layout(subG, seed=42)
nx.draw(subG, pos, node_color=node_colors, with_labels=True, node_size=400,
        font_size=8, edge_color='gray')
plt.title(f"Peer Network — Sample Classroom Cluster (n={len(target_component)})\nGreen=Low, Orange=Medium, Red=High risk")
cluster_png_name = f'Synthetic_school_classroom_cluster_{SUBSET_TAG}_v2.png'
plt.savefig(cluster_png_name, dpi=150, bbox_inches='tight')
plt.show()

# ============ PLOT 2: full-graph degree distribution ============
degrees = [d for _, d in G_full.degree()]
plt.figure(figsize=(8, 5))
plt.hist(degrees, bins=30, edgecolor='black')
plt.xlabel('Node degree (number of peer connections)')
plt.ylabel('Number of students')
plt.title(f'Degree Distribution — Full Peer Graph ({data.x.shape[0]} students)')
plt.axvline(sum(degrees)/len(degrees), color='red', linestyle='--', label=f'Mean degree = {sum(degrees)/len(degrees):.1f}')
plt.legend()
degree_png_name = f'Synthetic_school_data_degree_distribution_{SUBSET_TAG}_v2.png'
plt.savefig(degree_png_name, dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# ============================================================
# CELL 16 — sparse adjacency matrix + adjacency list export
# ============================================================
num_nodes = data.x.shape[0]
edge_index_np = data.edge_index.numpy()

adj_sparse = coo_matrix(
    (np.ones(edge_index_np.shape[1]), (edge_index_np[0], edge_index_np[1])),
    shape=(num_nodes, num_nodes)
)

print("Sparse adjacency matrix shape:", adj_sparse.shape)
print("Non-zero entries (directed edges incl. self-loops):", adj_sparse.nnz)

adj_list = {i: [] for i in range(num_nodes)}
for src, tgt in zip(edge_index_np[0], edge_index_np[1]):
    if src != tgt:
        adj_list[src].append(int(tgt))

sample_student = 0
print(f"\nStudent {sample_student} is connected to {len(adj_list[sample_student])} peers:")
print(adj_list[sample_student][:10], "..." if len(adj_list[sample_student]) > 10 else "")

adj_matrix_name = f'Synthetic_adjacency_matrix_{SUBSET_TAG}.npz'
adj_list_name = f'Synthetic_adjacency_list_{SUBSET_TAG}_v.pkl'

save_npz(adj_matrix_name, adj_sparse.tocsr())
with open(adj_list_name, 'wb') as f:
    pickle.dump(adj_list, f)

print("\nSaved both files")

In [ ]:
# ============================================================
# CELL 17 — final downloads (adjacency artifacts + plots)
# ============================================================
for fname in [adj_matrix_name, adj_list_name, cluster_png_name, degree_png_name]:
    files.download(fname)